In [73]:
import re
import string
from keybert import KeyBERT
from transformers import AutoTokenizer

In [108]:
text = 'Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Florida. He has appeared in numerous films, television shows, and theater productions throughout his career, which began in the late 1970s.'
tokenizer_path = 'databricks/dolly-v2-3b'
# tokenizer_path = 'bigscience/bloomz-560m'
# tokenizer_path = 'meta-llama/Llama-2-7b-chat-hf'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
int_tokens = tokenizer.encode(text)
tokens = [tokenizer.decode(t) for t in int_tokens]

In [109]:
pieces = [p.strip() for p in re.split('\n|\.|\?|\!', text) if len(p.strip()) > 0]
pieces

['Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Florida',
 'He has appeared in numerous films, television shows, and theater productions throughout his career, which began in the late 1970s']

In [110]:
thrs = 0.1
kw_model = KeyBERT()
kw_res = kw_model.extract_keywords(pieces, top_n=max(len(s) for s in pieces))
kw_res = [[(word, val) for word, val in piece_res if val > thrs] for piece_res in kw_res]

In [111]:
kw_res

[[('flaherty', 0.5644),
  ('lanny', 0.4359),
  ('pensacola', 0.3711),
  ('florida', 0.3069),
  ('actor', 0.2808),
  ('born', 0.2635),
  ('1949', 0.2382),
  ('december', 0.217),
  ('18', 0.1852),
  ('american', 0.1036)],
 [('1970s', 0.3781),
  ('productions', 0.328),
  ('shows', 0.2888),
  ('theater', 0.2778),
  ('films', 0.2609),
  ('television', 0.2131),
  ('career', 0.1031)]]

In [112]:
last = 0
marker_str = ''
for i in range(len(pieces)):
    while not text[last:].startswith(pieces[i]):
        last += 1
        marker_str += ' '
    words = [w.strip(string.punctuation) for w in pieces[i].split()]
    for w in words:
        while not text[last:].startswith(w):
            last += 1
            marker_str += ' '
        last += len(w)
        if w.lower() in [w for w, _ in kw_res[i]]:
            marker_str += '^' * len(w)
        else:
            marker_str += ' ' * len(w)
marker_str += ' ' * (len(text) - last)

In [113]:
last = 0
is_keyword = []
for t in tokens:
    if t == '<s>':
        is_keyword.append(False)
        continue
    tok = t.strip()
    while not text[last:].startswith(tok):
        last += 1
    is_keyword.append(marker_str[last:last + len(tok)] == '^' * len(tok))
    last += len(tok)

In [107]:
print(text[:86])
print(marker_str[:86])

Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Florida. 
^^^^^ ^^^^^^^^       ^^^^^^^^ ^^^^^ ^^^^    ^^^^^^^^ ^^  ^^^^     ^^^^^^^^^  ^^^^^^^  


In [71]:
print(text[86:186])
print(marker_str[86:186])

He has appeared in numerous films, television shows, and theater productions throughout his career, 
                            ^^^^^  ^^^^^^^^^^ ^^^^^      ^^^^^^^ ^^^^^^^^^^^                ^^^^^^  


In [72]:
print(text[186:])
print(marker_str[186:])

which began in the late 1970s.
                        ^^^^^ 


In [120]:
text = 'Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Florida. He has appeared in numerous films, television shows, and theater productions throughout his career, which began in the late 1970s.'

In [123]:
from factscore.atomic_facts import AtomicFactGenerator
af_generator = AtomicFactGenerator(
            key_path='/Users/ekaterinafadeeva/Downloads/openai_key.txt',
            demon_dir='/Users/ekaterinafadeeva/Downloads/demos',
            gpt3_cache_file='/Users/ekaterinafadeeva/Downloads/InstructGPT.pkl')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ekaterinafadeeva/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [124]:
curr_afs, _ = af_generator.run(text)

In [129]:
curr_afs

[('Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Florida.',
  ['Lanny Flaherty is an American actor.',
   'Lanny Flaherty was born on December 18, 1949.',
   'Lanny Flaherty was born in Pensacola, Florida.']),
 ('He has appeared in numerous films, television shows, and theater productions throughout his career, which began in the late 1970s.',
  ['He has appeared in numerous films.',
   'He has appeared in numerous television shows.',
   'He has appeared in numerous theater productions.',
   'He has appeared in numerous films, television shows, and theater productions.',
   'His career began in the late 1970s.'])]

In [130]:
match_strings = [
    ['                     ^^^^^^^^ ^^^^^                                                  ',
     '                                    ^^^^ ^^ ^^^^^^^^ ^^  ^^^^                        ',
     '                                                                  ^^^^^^^^^  ^^^^^^^ '],
    ['                            ^^^^^                                                                                                 ',
     '                                   ^^^^^^^^^^ ^^^^^                                                                               ',
     '                                                         ^^^^^^^ ^^^^^^^^^^^                                                      ',
     '                                                                                                                                  ',
     '                                                                                                          ^^^^^ ^^ ^^^ ^^^^ ^^^^^ ']
]

In [139]:
last = 0
new_class = 0
letter_classes = []
for (sent, _), matches in zip(curr_afs, match_strings):
    while not text[last:].lower().startswith(sent.lower()):
        last += 1
        letter_classes += [None]
    cur_classes = [None for _ in sent]
    for match_str in matches:
        assert len(match_str) == len(sent)
        for i in range(len(sent)):
            if match_str[i] == '^':
                assert cur_classes[i] is None
                cur_classes[i] = new_class
        new_class += 1
    letter_classes += cur_classes
    last += len(cur_classes)
letter_classes += [None for _ in range(len(text) - last)]

In [140]:
for c, cl in zip(text, letter_classes):
    print(c, cl)

L None
a None
n None
n None
y None
  None
F None
l None
a None
h None
e None
r None
t None
y None
  None
i None
s None
  None
a None
n None
  None
A 0
m 0
e 0
r 0
i 0
c 0
a 0
n 0
  None
a 0
c 0
t 0
o 0
r 0
  None
b 1
o 1
r 1
n 1
  None
o 1
n 1
  None
D 1
e 1
c 1
e 1
m 1
b 1
e 1
r 1
  None
1 1
8 1
, None
  None
1 1
9 1
4 1
9 1
, None
  None
i None
n None
  None
P 2
e 2
n 2
s 2
a 2
c 2
o 2
l 2
a 2
, None
  None
F 2
l 2
o 2
r 2
i 2
d 2
a 2
. None
  None
H None
e None
  None
h None
a None
s None
  None
a None
p None
p None
e None
a None
r None
e None
d None
  None
i None
n None
  None
n None
u None
m None
e None
r None
o None
u None
s None
  None
f 3
i 3
l 3
m 3
s 3
, None
  None
t 4
e 4
l 4
e 4
v 4
i 4
s 4
i 4
o 4
n 4
  None
s 4
h 4
o 4
w 4
s 4
, None
  None
a None
n None
d None
  None
t 5
h 5
e 5
a 5
t 5
e 5
r 5
  None
p 5
r 5
o 5
d 5
u 5
c 5
t 5
i 5
o 5
n 5
s 5
  None
t None
h None
r None
o None
u None
g None
h None
o None
u None
t None
  None
h None
i None
s None
  None
c None
a None
r

In [1]:
from lm_polygraph.token_clusterer import AtomicClusterer, KeyWordMatcher
from transformers import AutoTokenizer

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ekaterinafadeeva/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
tokenizer_path = 'databricks/dolly-v2-3b'
# tokenizer_path = 'bigscience/bloomz-560m'
# tokenizer_path = 'meta-llama/Llama-2-7b-chat-hf'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
cl = AtomicClusterer(KeyWordMatcher(verbose=True))

In [3]:
text = 'Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Florida.'
int_tokens = tokenizer.encode(text)
str_tokens = [tokenizer.decode([i]) for i in int_tokens]

classes = cl.cluster(text, str_tokens)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Match
	text: Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Florida.
	atoms:
		Lanny Flaherty is an American actor.
		Lanny Flaherty was born on December 18, 1949.
		Lanny Flaherty was born in Pensacola, Florida.
Keywords:
	 ['flaherty', 'lanny', 'actor', 'american']
	 ['flaherty', 'lanny', '1949', 'born', 'december', '18']
	 ['flaherty', 'pensacola', 'lanny', 'florida', 'born']
Matches:

Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Florida.
                     ^^^^^^^^ ^^^^^                                                  

Lanny Flaherty is an American actor born on December 18, 1949, in Pensacola, Flor